# electoral_sim: Quick Start

This notebook shows:
1. How to load a single scenario and run all electoral systems on it
2. How to load multiple scenarios and run them in one go

In [10]:
import numpy as np
from electoral_sim.scenario import load_scenario, load_all_scenarios
from electoral_sim.systems import get_all_systems
from electoral_sim.metrics import run_simulation

## 1. Load a single scenario and run

We load the Polarized Bimodal scenario — a two-cluster electorate analogous to contemporary USA or Brexit-era UK — and run all nine standard electoral systems on it.

In [11]:
rng = np.random.default_rng(42)

config, electorate, candidates = load_scenario(
    "../configs/scenarios/02_polarized_bimodal.yaml",
    rng=rng
)


print(f"Scenario : {config['name']}")
print(f"Analogue : {config['real_world_analog']}")
print(f"Voters    : {electorate.n_voters}")
print(f"Candidates: {candidates.labels}")

Scenario : Polarized Bimodal
Analogue : Contemporary USA, Brexit-era UK
Voters    : 5000
Candidates: ['Far-Left', 'Left', 'Center', 'Right', 'Far-Right']


In [12]:
systems = get_all_systems(rng=rng)
metrics = run_simulation(electorate, candidates, systems)

print(f"{'System':<35} {'d_median':>10} {'majority_sat':>14}")
print("-" * 62)
for m in sorted(metrics, key=lambda x: x.distance_to_median):
    print(f"{m.system_name:<35} {m.distance_to_median:>10.4f} {m.majority_satisfaction:>14.3f}")

System                                d_median   majority_sat
--------------------------------------------------------------
Borda Count                             0.0332          0.163
Score Voting                            0.0332          0.163
Condorcet (Schulze)                     0.0332          0.163
Party-List PR (D'Hondt, 100 seats)      0.0332          0.163
Plurality (FPTP)                        0.2218          0.290
Two-Round Runoff                        0.2218          0.290
Instant Runoff (IRV)                    0.2218          0.290
Approval Voting                         0.2218          0.290
Mixed Member Proportional (MMP)         0.2218          0.290


## 2. Load multiple scenarios and run

`load_all_scenarios` reads every YAML file in a directory and returns a list of `(config, electorate, candidates)` tuples. We loop over them and print the best-performing system in each scenario.

In [14]:
rng = np.random.default_rng(42)

scenarios = load_all_scenarios("../configs/scenarios/", rng=rng)
systems = get_all_systems(rng=rng)

print(f"Loaded {len(scenarios)} scenarios\n")

for config, electorate, candidates in scenarios:
    metrics = run_simulation(electorate, candidates, systems)
    best = min(metrics, key=lambda x: x.distance_to_median)
    print(f"{config['name']:<35}  best: {best.system_name:<30}  d_median={best.distance_to_median:.4f}")

Loaded 8 scenarios

Unimodal Consensus                   best: Borda Count                     d_median=0.0492
Polarized Bimodal                    best: Borda Count                     d_median=0.0278
Multimodal Fragmented                best: Plurality (FPTP)                d_median=0.1344
Dominant Party with Minorities       best: Plurality (FPTP)                d_median=0.0962
Asymmetric Skewed                    best: Plurality (FPTP)                d_median=0.0417
Two-Party Symmetric Polarized        best: Two-Round Runoff                d_median=0.1309
Two-Party Asymmetric Centrist Majority  best: Plurality (FPTP)                d_median=0.1204
Two-Party Dominant Left              best: Instant Runoff (IRV)            d_median=0.0134


To inspect the full results for any scenario, sort and print the full metrics list as shown in Section 1.